# Figure 5: CellChat, paired SPP1, and spatial hotspots

All-gene input preparation, broad CellChat screening, paired SPP1 expression, tumor-only communication, and spatial hotspots. Shared pseudobulk differential expression is supplied by the Figure 2 analysis.


## 1. Prepare the all-gene CellChat input


In [ ]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
from scipy import io as spio

PROJECT_ROOT = Path("/mnt/disk18t/lr_xcy/riku/kirc_205")
METHOD_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_communication/official_methods/cellchat_r_all_genes")
INPUTS = METHOD_DIR / "inputs"
OUTPUTS = METHOD_DIR / "outputs"
INPUT_H5AD = INPUTS / "adata_anno_cell_subtype_re.h5ad"
SUBTYPE_INPUT_H5AD = INPUTS / "subtype_full_lr_input.h5ad"
SUBTYPE_PAIRS_CSV = INPUTS / "subtype_all_pairs.csv"
CM_EPI_PAIRS_CSV = INPUTS / "cm_epi_bidirectional_pairs.csv"
CM_NODE_SET_CSV = INPUTS / "balanced_joint_cm_reference_node_sets_after_edge_threshold.csv"
CM_NODE_MAPPING_CSV = INPUTS / "cm_node_mapping_for_subtype_aggregation.csv"
EPI_SUBTYPES_CSV = INPUTS / "epi_subtypes.csv"
GROUP_KEY = "ccc_group"
SUBTYPE_KEY = "cell_subtype"


def read_required_csv(path: Path, required_cols: set[str]) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required fixed input CSV: {path}")
    df = pd.read_csv(path)
    if not required_cols.issubset(df.columns):
        raise RuntimeError(f"{path} must contain {sorted(required_cols)}")
    return df


def load_cm_node_mapping(valid_subtypes: set[str] | None = None) -> pd.DataFrame:
    source = CM_NODE_MAPPING_CSV if CM_NODE_MAPPING_CSV.exists() else CM_NODE_SET_CSV
    if not source.exists():
        raise FileNotFoundError(f"Missing CM-node mapping file: {source}")
    mapping = pd.read_csv(source)
    required = {"CM", "reference_node_rank", "node"}
    if not required.issubset(mapping.columns):
        raise RuntimeError(f"{source} must contain {sorted(required)}")
    keep_cols = ["CM", "reference_node_rank", "node"]
    if "node_in_adata" in mapping.columns:
        keep_cols.append("node_in_adata")
    mapping = mapping[keep_cols].copy()
    mapping["CM"] = mapping["CM"].astype(str)
    mapping["node"] = mapping["node"].astype(str)
    mapping["reference_node_rank"] = pd.to_numeric(mapping["reference_node_rank"], errors="coerce").astype("Int64")
    mapping = mapping[mapping["CM"].str.startswith("joint_")].drop_duplicates(["CM", "node"])
    if "node_in_adata" in mapping.columns:
        mapping["node_in_adata"] = mapping["node_in_adata"].astype(str).str.lower().isin(["true", "1", "yes"])
    elif valid_subtypes is not None:
        mapping["node_in_adata"] = mapping["node"].isin(valid_subtypes)
    else:
        mapping["node_in_adata"] = True
    if valid_subtypes is not None:
        mapping["node_in_adata"] = mapping["node_in_adata"] & mapping["node"].isin(valid_subtypes)
    return mapping


def export_cellchat_input(adata_obj: ad.AnnData) -> None:
    out = INPUTS / "cellchat_input"
    out.mkdir(parents=True, exist_ok=True)
    x = adata_obj.X.T
    spio.mmwrite(out / "expression_genes_by_cells.mtx", x)
    pd.DataFrame({"gene": adata_obj.var_names}).to_csv(out / "genes.tsv", sep="	", index=False)
    pd.DataFrame({"cell": adata_obj.obs_names}).to_csv(out / "cells.tsv", sep="	", index=False)
    optional = [c for c in ["cell_type", "cell_subtype", "sample", "status"] if c in adata_obj.obs.columns]
    meta = adata_obj.obs[[GROUP_KEY] + optional].copy()
    meta.insert(0, "cell", adata_obj.obs_names)
    meta.to_csv(out / "metadata.tsv", sep="	", index=False)


def prepare_input(force: bool = False) -> None:
    INPUTS.mkdir(parents=True, exist_ok=True)
    cellchat_dir = INPUTS / "cellchat_input"
    required = [
        SUBTYPE_INPUT_H5AD,
        SUBTYPE_PAIRS_CSV,
        CM_EPI_PAIRS_CSV,
        CM_NODE_MAPPING_CSV,
        EPI_SUBTYPES_CSV,
        INPUTS / "group_counts.csv",
        cellchat_dir / "expression_genes_by_cells.mtx",
        cellchat_dir / "genes.tsv",
        cellchat_dir / "cells.tsv",
        cellchat_dir / "metadata.tsv",
    ]
    if all(p.exists() for p in required) and not force:
        return

    if not INPUT_H5AD.exists():
        raise FileNotFoundError(f"Missing input AnnData file: {INPUT_H5AD}")

    a = ad.read_h5ad(INPUT_H5AD, backed="r")
    obs = a.obs.copy()
    if SUBTYPE_KEY not in obs.columns:
        raise RuntimeError(f"{INPUT_H5AD} lacks obs['{SUBTYPE_KEY}']")
    obs[SUBTYPE_KEY] = obs[SUBTYPE_KEY].astype(str)

    all_subtypes = set(obs[SUBTYPE_KEY].unique())
    cm_mapping = load_cm_node_mapping(valid_subtypes=all_subtypes)
    missing_nodes = sorted(cm_mapping.loc[~cm_mapping["node_in_adata"], "node"].unique())
    if missing_nodes:
        pd.DataFrame({"missing_node": missing_nodes}).to_csv(INPUTS / "cm_nodes_missing_from_adata.csv", index=False)
    cm_mapping = cm_mapping[cm_mapping["node_in_adata"]].copy()
    cm_nodes = sorted(cm_mapping["node"].unique())
    epi_input = read_required_csv(EPI_SUBTYPES_CSV, {"Epi"})
    cm_epi_input = read_required_csv(CM_EPI_PAIRS_CSV, {"source", "target", "direction"})
    subtype_pairs_input = read_required_csv(SUBTYPE_PAIRS_CSV, {"source", "target"})
    read_required_csv(INPUTS / "group_counts.csv", {"group", "direction_class", "available_cells", "used_cells"})

    epis = [e for e in epi_input["Epi"].astype(str).tolist() if e in all_subtypes]
    cms = sorted(cm_mapping["CM"].unique())
    pair_groups = set(subtype_pairs_input["source"].astype(str)) | set(subtype_pairs_input["target"].astype(str))
    analysis_groups = sorted(pair_groups & all_subtypes)
    valid = obs[SUBTYPE_KEY].isin(analysis_groups)
    obs = obs.loc[valid].copy()

    selected_names: list[str] = []
    for group, idx in obs.groupby(SUBTYPE_KEY, observed=True).groups.items():
        names = np.array(list(idx), dtype=object)
        selected_names.extend(names.tolist())

    if a.raw is not None:
        b = a.raw[selected_names, :].to_adata().copy()
        expression_source = ".raw normalized/log expression"
    else:
        b = a[selected_names, :].to_memory().copy()
        expression_source = ".X expression because .raw was not available"

    b.obs[GROUP_KEY] = b.obs[SUBTYPE_KEY].astype(str)
    b.obs["direction_class"] = np.where(
        b.obs[GROUP_KEY].str.startswith("Epi_"),
        "Epi",
        np.where(b.obs[GROUP_KEY].isin(cm_nodes), "CM_node", "Other_subtype"),
    )
    b.uns["cm_groups"] = cms
    b.uns["epi_groups"] = epis
    b.uns["cm_nodes"] = cm_nodes
    b.uns["all_subtype_groups"] = analysis_groups
    b.uns["input_note"] = (
        f"All-gene real-cell CellChat input regenerated from {INPUT_H5AD.name}; all cells are grouped by cell_subtype "
        f"without downsampling; expression source is {expression_source}. No ligand-receptor gene prefilter is applied "
        "before exporting the CellChat expression matrix; R CellChat subsetData() uses CellChatDB.human downstream. "
        "Fixed CM/Epi pair, subtype-pair, epithelial-subtype, group-count, and CM-node mapping CSVs are copied from "
        "the original official CellChat workflow and are not regenerated by this all-gene input export. CM-Epi results "
        "are aggregated after subtype-subtype CellChat using only CM nodes marked node_in_adata=True in the mapping table."
    )
    b.write_h5ad(SUBTYPE_INPUT_H5AD)
    export_cellchat_input(b)

prepare_input(force=False)


## 2. Run CellChat and retain the two required analysis outputs


In [ ]:
%%script R
method_dir <- "/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_communication/official_methods/cellchat_r_all_genes"
input_dir <- file.path(method_dir, "inputs/cellchat_input")
output_dir <- file.path(method_dir, "outputs")
pairs_csv <- file.path(method_dir, "inputs/subtype_all_pairs.csv")
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
Sys.setenv(TMPDIR = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/cm_epi_communication_r_tmp_official_ccc")
Sys.setenv(R_LIBS_USER = "/mnt/disk18t/lr_xcy/riku/codex_research/envs/r-cellchat")
.libPaths(c(Sys.getenv("R_LIBS_USER"), .libPaths()))
suppressPackageStartupMessages({
  library(CellChat)
  library(Matrix)
  library(future)
})

cellchat_workers <- as.integer(Sys.getenv("CELLCHAT_WORKERS", "8"))
if (is.na(cellchat_workers) || cellchat_workers < 1) {
  cellchat_workers <- 1
}
options(future.globals.maxSize = 100 * 1024^3)
future::plan("multisession", workers = cellchat_workers)

expr <- readMM(file.path(input_dir, "expression_genes_by_cells.mtx"))
genes <- read.delim(file.path(input_dir, "genes.tsv"), stringsAsFactors = FALSE)
cells <- read.delim(file.path(input_dir, "cells.tsv"), stringsAsFactors = FALSE)
meta <- read.delim(file.path(input_dir, "metadata.tsv"), stringsAsFactors = FALSE)
rownames(expr) <- make.unique(genes$gene)
colnames(expr) <- cells$cell
rownames(meta) <- meta$cell
meta$ccc_group <- factor(meta$ccc_group)

cellchat <- createCellChat(object = expr, meta = meta, group.by = "ccc_group")
cellchat@DB <- CellChatDB.human
cellchat <- subsetData(cellchat)
cellchat <- identifyOverExpressedGenes(cellchat)
cellchat <- identifyOverExpressedInteractions(cellchat)
cellchat <- computeCommunProb(cellchat, type = "triMean", nboot = 100, seed.use = 1337)
cellchat <- filterCommunication(cellchat, min.cells = 10)
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)
saveRDS(cellchat, file.path(output_dir, "cellchat_object.rds"))

all_communication <- subsetCommunication(cellchat)
pairs <- read.csv(pairs_csv, stringsAsFactors = FALSE)
selected_communication <- merge(
  all_communication,
  pairs[, c("source", "target")],
  by = c("source", "target")
)
write.csv(
  selected_communication,
  file.path(output_dir, "official_cellchat_subtype_lr_results.csv"),
  row.names = FALSE
)

future::plan("sequential")


## 3. Prepare the Epi_JUN-to-CM1 bubble-plot CSV


In [ ]:

METHOD_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_communication/official_methods/cellchat_r_all_genes")
OUTPUT_DIR = METHOD_DIR / "outputs"
MAPPING_CSV = METHOD_DIR / "inputs/cm_node_mapping_for_subtype_aggregation.csv"
CELLCHAT_LR_CSV = OUTPUT_DIR / "official_cellchat_subtype_lr_results.csv"
BUBBLE_DIR = OUTPUT_DIR / "claim_support/epi_jun_cm1_bubble"
BUBBLE_CSV = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_data.csv"

SOURCE = "Epi_JUN"
CM = "joint_01_sharedCM"
KEEP_LR_PAIRS = {
    "GDF15|TGFBR2",
    "MIF|CD74_CD44",
    "MIF|CD74_CXCR4",
    "SPP1|CD44",
    "SPP1|ITGA4_ITGB1",
    "SPP1|ITGAV_ITGB1",
    "SPP1|ITGAV_ITGB5",
    "TGFB1|TGFbR1_R2",
    "TNFSF9|TNFRSF9",
}


def load_cm_nodes() -> list[str]:
    mapping = pd.read_csv(MAPPING_CSV)
    if "node_in_adata" in mapping.columns:
        keep = mapping["node_in_adata"].astype(str).str.lower().isin(["true", "t", "1", "yes"])
        mapping = mapping.loc[keep].copy()
    selected = mapping.loc[mapping["CM"].astype(str).eq(CM)].copy()
    selected = selected.sort_values("reference_node_rank")
    return selected["node"].astype(str).drop_duplicates().tolist()


def make_epi_jun_cm1_bubble_data() -> None:
    communication = pd.read_csv(CELLCHAT_LR_CSV)
    communication["source"] = communication["source"].astype(str)
    communication["target"] = communication["target"].astype(str)
    communication["ligand"] = communication["ligand"].astype(str)
    communication["receptor"] = communication["receptor"].astype(str)
    communication["pval"] = pd.to_numeric(communication["pval"], errors="coerce")
    communication["communication_weight"] = pd.to_numeric(
        communication["communication_weight"]
        if "communication_weight" in communication.columns
        else communication["prob"],
        errors="coerce",
    ).fillna(0)
    communication["lr_pair"] = communication["ligand"] + "|" + communication["receptor"]

    bubble_data = communication.loc[
        communication["source"].eq(SOURCE)
        & communication["target"].isin(load_cm_nodes())
        & communication["pval"].le(0.05)
        & communication["annotation"].astype(str).eq("Secreted Signaling")
        & communication["lr_pair"].isin(KEEP_LR_PAIRS)
    ].copy()
    bubble_data = bubble_data.sort_values(["target", "lr_pair"]).reset_index(drop=True)
    BUBBLE_DIR.mkdir(parents=True, exist_ok=True)
    bubble_data.to_csv(BUBBLE_CSV, index=False)


make_epi_jun_cm1_bubble_data()


## 4. Draw the retained Figure 5B bubble plot


In [ ]:
import matplotlib

matplotlib.use("Agg")
matplotlib.rcParams["pdf.fonttype"] = 42
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcdefaults()
matplotlib.rcParams["pdf.fonttype"] = 42


METHOD_DIR = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_communication/official_methods/cellchat_r_all_genes")
BUBBLE_DIR = METHOD_DIR / "outputs/claim_support/epi_jun_cm1_bubble"
DATA_CSV = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_data.csv"
OUTPUT_PDF = BUBBLE_DIR / "cellchat_official_epi_jun_cm1_bubble_plot.pdf"


def plot_epi_jun_cm1_bubble() -> None:
    data = pd.read_csv(DATA_CSV)
    color_col = "prob" if "prob" in data.columns else "lr_probs"
    size_col = "communication_weight"
    data[color_col] = pd.to_numeric(data[color_col], errors="coerce").fillna(0)
    data[size_col] = pd.to_numeric(data[size_col], errors="coerce").fillna(0)
    data["lr_pair"] = data["lr_pair"].astype(str)
    data["target"] = data["target"].astype(str)

    target_order = data.groupby("target", observed=True)[size_col].sum().sort_values(ascending=False).index.tolist()
    pair_order = data.groupby("lr_pair", observed=True)[size_col].max().sort_values(ascending=True).index.tolist()
    x_map = {value: index for index, value in enumerate(target_order)}
    y_map = {value: index for index, value in enumerate(pair_order)}

    weights = data[size_col].to_numpy(dtype=float)
    max_weight = np.nanmax(weights) if len(weights) else 0
    sizes = 18 + 210 * np.sqrt(weights / max_weight) if max_weight > 0 else np.repeat(28, len(data))
    fig_width = min(max(7.5, 0.55 * len(target_order) + 3.0), 11.0)
    fig_height = min(max(4.35, 0.20 * len(pair_order) + 2.1), 7.6)

    fig, axis = plt.subplots(figsize=(fig_width, fig_height))
    scatter = axis.scatter(
        data["target"].map(x_map).to_numpy(),
        data["lr_pair"].map(y_map).to_numpy(),
        c=data[color_col].to_numpy(dtype=float),
        s=sizes,
        cmap="viridis",
        edgecolors="none",
        linewidths=0,
        alpha=0.9,
    )
    axis.set_xticks(range(len(target_order)))
    axis.set_xticklabels(target_order, rotation=45, ha="right", fontsize=7)
    axis.set_yticks(range(len(pair_order)))
    axis.set_yticklabels(pair_order, fontsize=6)
    axis.set_xlabel("CM component subtype", fontsize=8)
    axis.set_ylabel("Ligand-receptor pair", fontsize=8)
    axis.set_title("CellChat official: Epi_JUN to CM1", fontsize=10)
    axis.grid(True, axis="both", color="0.88", linewidth=0.5)
    axis.set_axisbelow(True)
    colorbar = fig.colorbar(scatter, ax=axis, fraction=0.025, pad=0.01)
    colorbar.set_label(color_col, fontsize=7)
    colorbar.ax.tick_params(labelsize=6)

    if np.any(weights > 0):
        legend_values = np.quantile(weights[weights > 0], [0.33, 0.66, 1.0])
        handles = [
            plt.scatter(
                [], [], s=18 + 210 * np.sqrt(value / max_weight),
                color="0.55", edgecolors="none", linewidths=0,
            )
            for value in legend_values
        ]
        axis.legend(
            handles,
            [f"{value:.3f}" for value in legend_values],
            title=size_col,
            loc="upper right",
            bbox_to_anchor=(1.0, 1.02),
            fontsize=6,
            title_fontsize=6,
            frameon=True,
        )
    fig.tight_layout()
    fig.savefig(OUTPUT_PDF, bbox_inches="tight")
    plt.close(fig)


plot_epi_jun_cm1_bubble()


## 5. Paired ligand effects and SPP1 expression

The paired sample list and all-gene edgeR results are reused from Figure 2; SPP1 mean log-normalized expression is calculated for the same 41 tumor pairs.


In [ ]:
from scipy import sparse

FINAL_SOURCE = METHOD_DIR / "outputs/final_figure_source_data"
FINAL_SOURCE.mkdir(parents=True, exist_ok=True)
FIG2_SOURCE_WORK = PROJECT_ROOT / "review_revision_20260828_li/results/tumor_epithelial_marker_publication_figure_repositioned/source_data_work"
FIG2_EDGER = FIG2_SOURCE_WORK / "panel_b_edger_results.csv.gz"
FIG2_METADATA = FIG2_SOURCE_WORK / "panel_b_pseudobulk_metadata.csv"
for required in [FIG2_EDGER, FIG2_METADATA]:
    if not required.exists():
        raise FileNotFoundError(
            f"Missing Figure 2 upstream result: {required}. "
            "Run the Figure 2 pseudobulk preparation and run_fig2_jun_vim_edger() first."
        )

metadata = pd.read_csv(FIG2_METADATA)
STATES = ["Epi_JUN", "Epi_VIM"]
if metadata.duplicated(["sample", "state"]).any():
    raise ValueError("Duplicate sample-state libraries in Figure 2 metadata")
cell_counts = metadata.pivot(index="sample", columns="state", values="n_cells").reindex(columns=STATES)
paired = cell_counts.index[(cell_counts >= 20).all(axis=1)].astype(str).tolist()
if len(paired) != 41 or len(metadata) != 82:
    raise ValueError("Expected the 41 paired tumors used for Figure 2")
edger = pd.read_csv(FIG2_EDGER)
if edger["gene"].duplicated().any() or not edger["n_paired_tumors"].eq(41).all():
    raise ValueError("Unexpected Figure 2 edgeR result")
if not edger["contrast"].eq("Epi_JUN_minus_Epi_VIM").all():
    raise ValueError("Unexpected edgeR contrast")
edger.to_csv(FINAL_SOURCE / "Figure5A_all_gene_paired_edgeR.csv.gz", index=False, compression="gzip")
candidates = ["SPP1", "ANGPTL4", "PROS1", "TGFA", "TNFSF9", "MIF", "GDF15", "TGFB1", "MDK"]
ligands = edger.loc[edger["gene"].isin(candidates)].copy()
if len(ligands) != len(candidates):
    raise ValueError("A candidate ligand is missing from the Figure 2 result")
ligands.to_csv(FINAL_SOURCE / "Figure5A_ligand_screen.csv", index=False)

adata_log = ad.read_h5ad(SUBTYPE_INPUT_H5AD, backed="r")
log_obs = adata_log.obs[["status", "sample", "series", GROUP_KEY]].astype(str)
log_keep = log_obs["status"].eq("tumor") & log_obs["sample"].isin(paired) & log_obs[GROUP_KEY].isin(STATES)
log_index = np.flatnonzero(log_keep.to_numpy())
values = adata_log.X[log_index, adata_log.var_names.get_loc("SPP1")]
if hasattr(values, "to_memory"):
    values = values.to_memory()
values = values.toarray().ravel() if sparse.issparse(values) else np.asarray(values).ravel()
spp1_long = log_obs.iloc[log_index][["sample", "series", GROUP_KEY]].copy()
spp1_long["SPP1"] = values
paired_spp1 = spp1_long.groupby(["sample", "series", GROUP_KEY], observed=True)["SPP1"].mean().unstack(GROUP_KEY).dropna(subset=STATES).reset_index()
paired_spp1["Epi_JUN_minus_Epi_VIM"] = paired_spp1["Epi_JUN"] - paired_spp1["Epi_VIM"]
if len(paired_spp1) != 41 or set(paired_spp1["sample"]) != set(paired):
    raise ValueError("The SPP1 expression pairs do not match Figure 2")
paired_spp1.to_csv(FINAL_SOURCE / "Figure5B_paired_SPP1_expression_source_data.csv", index=False)
adata_log.file.close()


In [ ]:
import json
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from scipy.stats import wilcoxon

FIGURE_ROOT = PROJECT_ROOT / "review_revision_20260828_li/results"
LIGAND_SOURCE = FINAL_SOURCE / "Figure5A_ligand_screen.csv"
LIGAND_OUT = FIGURE_ROOT / "figure5a_ranked_ligand_screen_all_fdr_muted"
PAIRED_SOURCE = FINAL_SOURCE / "Figure5B_paired_SPP1_expression_source_data.csv"
PAIRED_OUT = FIGURE_ROOT / "figure5b_paired_spp1_significance_gray_linetype_compact_width_groups"
FIGURE_WIDTH, X_PADDING, GROUP_GAP = 6.65, 0.18, 0.84
OUTPUT_STEM = "Figure5B_paired_SPP1_change_paired_Wilcoxon_gray_solid_dashed_compact_width_groups"

def format_wilcoxon_p(value):
    mantissa, exponent = f"{value:.2e}".split("e")
    return rf"Paired Wilcoxon $P = {mantissa} \times 10^{{{int(exponent)}}}$"

LIGAND_COLORS = {'highlight': '#B84A4A', 'neutral': '#6F7880', 'dark': '#263238', 'mid': '#66717B', 'grid': '#E5EAED'}

def ligand_configure() -> None:
    regular = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF')
    bold = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF')
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    mpl.rcParams.update({'font.family': 'Arial' if regular.exists() else 'DejaVu Sans', 'font.size': 8.5, 'axes.labelsize': 9.0, 'xtick.labelsize': 8.0, 'ytick.labelsize': 8.0, 'axes.linewidth': 0.75, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})

def ligand_header(fig: plt.Figure) -> None:
    fig.text(0.035, 0.965, 'A', fontsize=16, fontweight='bold', color=LIGAND_COLORS['dark'], va='top')
    fig.text(0.095, 0.965, 'SPP1 is the top Epi_JUN-enriched candidate ligand', fontsize=12.5, fontweight='bold', color=LIGAND_COLORS['dark'], va='top')
    fig.text(0.095, 0.91, 'Paired tumor-only pseudobulk; 41 tumors', fontsize=8.5, color=LIGAND_COLORS['mid'], va='top')

def ligand_save(fig: plt.Figure, stem_name: str) -> None:
    LIGAND_OUT.mkdir(parents=True, exist_ok=True)
    stem = LIGAND_OUT / stem_name
    fig.savefig(stem.with_suffix('.pdf'), dpi=450, bbox_inches='tight')
    plt.close(fig)

def plot_ligand_screen() -> None:
    ligand_configure()
    data = pd.read_csv(LIGAND_SOURCE).sort_values('logFC', ascending=False).reset_index(drop=True)
    LIGAND_OUT.mkdir(parents=True, exist_ok=True)
    data.to_csv(LIGAND_OUT / 'Figure5A_ligand_screen_source_data.csv', index=False)
    (fig, ax) = plt.subplots(figsize=(7.09, 4.45))
    fig.subplots_adjust(left=0.19, right=0.94, bottom=0.17, top=0.8)
    ligand_header(fig)
    y = np.arange(len(data))
    for (idx, row) in data.iterrows():
        is_spp1 = row['gene'] == 'SPP1'
        color = LIGAND_COLORS['highlight'] if is_spp1 else LIGAND_COLORS['neutral']
        width = 3.0 if is_spp1 else 1.8
        size = 82 if is_spp1 else 50
        ax.plot([0, row['logFC']], [idx, idx], color=color, lw=width, solid_capstyle='butt')
        ax.scatter(row['logFC'], idx, s=size, color=color, edgecolor='white', linewidth=0.65, zorder=3)
        positive = row['logFC'] >= 0
        x_offset = 0.025 if positive else -0.025
        ax.text(row['logFC'] + x_offset, idx, f"FDR {row['FDR']:.1e}", va='center', ha='left' if positive else 'right', color=color, fontsize=7.3, fontweight='bold' if is_spp1 else 'normal')
    ax.axvline(0, color='#59636A', lw=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(data['gene'])
    ax.invert_yaxis()
    for label in ax.get_yticklabels():
        if label.get_text() == 'SPP1':
            label.set_color(LIGAND_COLORS['highlight'])
            label.set_fontweight('bold')
    x_min = min(-0.66, float(data['logFC'].min()) - 0.16)
    x_max = max(0.78, float(data['logFC'].max()) + 0.16)
    ax.set_xlim(x_min, x_max)
    ax.set_xlabel('Epi_JUN − Epi_VIM log2 fold change')
    ax.text(0.01, 1.025, '←  Epi_VIM higher', transform=ax.transAxes, color=LIGAND_COLORS['neutral'], fontweight='bold', ha='left', va='bottom', clip_on=False)
    ax.text(0.99, 1.025, 'Epi_JUN higher  →', transform=ax.transAxes, color=LIGAND_COLORS['neutral'], fontweight='bold', ha='right', va='bottom', clip_on=False)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', color=LIGAND_COLORS['grid'], lw=0.65)
    ax.set_axisbelow(True)
    ligand_save(fig, 'Figure5A_ranked_ligand_screen_all_FDR_muted')

PAIRED_COLORS = {'vim': '#2A9D8F', 'jun': '#E67E22', 'vim_dark': '#176D65', 'jun_dark': '#B85C13', 'line': '#AEB7BC', 'dark': '#263238', 'mid': '#66717B', 'grid': '#E5EAED'}

def paired_configure() -> None:
    regular = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF')
    bold = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF')
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    mpl.rcParams.update({'font.family': 'Arial' if regular.exists() else 'DejaVu Sans', 'font.size': 8.5, 'axes.titlesize': 9.5, 'axes.labelsize': 9.0, 'xtick.labelsize': 8.0, 'ytick.labelsize': 8.0, 'axes.linewidth': 0.75, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})

def paired_header(fig: plt.Figure, letter: str, title: str, subtitle: str) -> None:
    fig.text(0.035, 0.965, letter, fontsize=16, fontweight='bold', color=PAIRED_COLORS['dark'], va='top')
    fig.text(0.095, 0.965, title, fontsize=12.5, fontweight='bold', color=PAIRED_COLORS['dark'], va='top')
    fig.text(0.095, 0.91, subtitle, fontsize=8.5, color=PAIRED_COLORS['mid'], va='top')

def paired_save(fig: plt.Figure, name: str) -> None:
    PAIRED_OUT.mkdir(parents=True, exist_ok=True)
    stem = PAIRED_OUT / name
    fig.savefig(stem.with_suffix('.pdf'), dpi=450, bbox_inches='tight')
    plt.close(fig)

def plot_paired_spp1() -> None:
    paired_configure()
    PAIRED_OUT.mkdir(parents=True, exist_ok=True)
    data = pd.read_csv(PAIRED_SOURCE).sort_values('Epi_JUN_minus_Epi_VIM').reset_index(drop=True)
    data.to_csv(PAIRED_OUT / 'Figure5B_paired_SPP1_expression_source_data.csv', index=False)
    test = wilcoxon(data['Epi_JUN'], data['Epi_VIM'], alternative='two-sided')
    positive = int((data['Epi_JUN_minus_Epi_VIM'] > 0).sum())
    statistics = {'test': 'two-sided paired Wilcoxon signed-rank test', 'n_tumors': int(len(data)), 'statistic': float(test.statistic), 'p_value': float(test.pvalue), 'tumors_with_Epi_JUN_higher': positive, 'median_Epi_VIM': float(data['Epi_VIM'].median()), 'median_Epi_JUN': float(data['Epi_JUN'].median()), 'median_paired_difference': float(data['Epi_JUN_minus_Epi_VIM'].median())}
    (PAIRED_OUT / 'Figure5B_paired_SPP1_statistics.json').write_text(json.dumps(statistics, indent=2), encoding='utf-8')
    (fig, ax) = plt.subplots(figsize=(FIGURE_WIDTH, 4.55))
    fig.subplots_adjust(left=0.13, right=0.95, bottom=0.16, top=0.8)
    paired_header(fig, 'B', 'SPP1 expression is higher in Epi_JUN than Epi_VIM', f'Paired tumour-only comparison across {len(data)} tumours')
    x_left = (1.0 - GROUP_GAP) / 2.0
    x_right = (1.0 + GROUP_GAP) / 2.0
    line_color = '#929CA2'
    for (_, row) in data.iterrows():
        increase = row['Epi_JUN_minus_Epi_VIM'] > 0
        ax.plot([x_left, x_right], [row['Epi_VIM'], row['Epi_JUN']], color=line_color, lw=0.95 if increase else 1.25, linestyle='-' if increase else (0, (4.0, 2.4)), alpha=0.56 if increase else 0.95, zorder=1 if increase else 2)
    ax.scatter(np.full(len(data), x_left), data['Epi_VIM'], s=28, color=PAIRED_COLORS['vim'], edgecolor='white', linewidth=0.55, zorder=3)
    ax.scatter(np.full(len(data), x_right), data['Epi_JUN'], s=31, color=PAIRED_COLORS['jun'], edgecolor='white', linewidth=0.55, zorder=3)
    med_vim = float(data['Epi_VIM'].median())
    med_jun = float(data['Epi_JUN'].median())
    ax.plot([x_left - 0.1, x_left + 0.1], [med_vim, med_vim], color=PAIRED_COLORS['vim_dark'], lw=4.0, solid_capstyle='butt')
    ax.plot([x_right - 0.1, x_right + 0.1], [med_jun, med_jun], color=PAIRED_COLORS['jun_dark'], lw=4.0, solid_capstyle='butt')
    y_max = float(max(data['Epi_VIM'].max(), data['Epi_JUN'].max()))
    bracket_y = y_max + 0.1
    bracket_h = 0.07
    ax.plot([x_left, x_left, x_right, x_right], [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y], color=PAIRED_COLORS['dark'], lw=0.9, clip_on=False)
    ax.text((x_left + x_right) / 2.0, bracket_y + bracket_h + 0.035, format_wilcoxon_p(test.pvalue), ha='center', va='bottom', color=PAIRED_COLORS['dark'], fontsize=8.8)
    direction_handles = [Line2D([0], [0], color=line_color, lw=1.4, linestyle='-', label='Increase'), Line2D([0], [0], color=line_color, lw=1.4, linestyle=(0, (4.0, 2.4)), label='Decrease')]
    fig.legend(handles=direction_handles, loc='upper left', bbox_to_anchor=(0.095, 0.865), frameon=False, ncol=2, handlelength=2.2, columnspacing=1.2, borderaxespad=0.0)
    ax.set_xlim(-X_PADDING, 1 + X_PADDING)
    ax.set_ylim(0.1, bracket_y + 0.34)
    ax.set_xticks([x_left, x_right])
    ax.set_xticklabels(['Epi_VIM', 'Epi_JUN'], fontsize=10.5, fontweight='bold')
    ax.get_xticklabels()[0].set_color(PAIRED_COLORS['vim'])
    ax.get_xticklabels()[1].set_color(PAIRED_COLORS['jun'])
    ax.set_ylabel('Mean log-normalized SPP1 expression')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', color=PAIRED_COLORS['grid'], lw=0.65)
    ax.set_axisbelow(True)
    paired_save(fig, OUTPUT_STEM)

plot_ligand_screen()
plot_paired_spp1()


## 6. Final Figure 5C: tumour-only balanced all-gene CellChat

The all-gene CellChat object generated above is restricted to Epi_JUN, Epi_VIM and the ten CM1 nodes. Up to 100 cells are sampled per `sample × group` stratum before recomputing candidate-ligand communication probabilities.


In [ ]:
%%script R
Sys.setenv(R_LIBS_USER="/mnt/disk18t/lr_xcy/riku/codex_research/envs/r-cellchat")
.libPaths(c(Sys.getenv("R_LIBS_USER"),.libPaths()))
suppressPackageStartupMessages({library(CellChat); library(future)})
root <- "/mnt/disk18t/lr_xcy/riku/kirc_205"
source_rds <- "/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/cm_epi_communication/official_methods/cellchat_r_all_genes/outputs/cellchat_object.rds"
out_dir <- file.path(root,"review_revision_20260828_li/results/figure5c_cellchat_all_genes/tumor_only_cellchat")
dir.create(out_dir,recursive=TRUE,showWarnings=FALSE)
cm1_nodes <- c("T_IL7R","T_GNLY","T_KLRB1","T_FOXP3","Mast_FYN_high","S_DCN","T_TOX","T_GZMK","NK_KLRF1","NK_XCL1")
source_states <- c("Epi_JUN","Epi_VIM"); groups_use <- c(source_states,cm1_nodes)
candidate_ligands <- c("SPP1","MIF","GDF15","TGFB1","TNFSF9","ANGPTL4","PROS1","TGFA","MDK")
cap <- 100L; seed <- 20260820L; set.seed(seed)
options(future.globals.maxSize=30*1024^3); future::plan("multicore",workers=4L)
source_object <- readRDS(source_rds); meta <- source_object@meta
meta$ccc_group <- as.character(meta$ccc_group); meta$sample <- as.character(meta$sample); meta$status <- as.character(meta$status)
eligible <- rownames(meta)[meta$status=="tumor" & meta$ccc_group %in% groups_use]
eligible_meta <- meta[eligible,,drop=FALSE]
strata <- split(rownames(eligible_meta),interaction(eligible_meta$ccc_group,eligible_meta$sample,drop=TRUE))
selected <- sort(unique(unlist(lapply(strata,function(cells) if(length(cells)<=cap) cells else sample(cells,cap)),use.names=FALSE)))
selection <- eligible_meta[selected,c("ccc_group","sample","status"),drop=FALSE]; selection$cell <- rownames(selection)
write.csv(selection[,c("cell","ccc_group","sample","status")],gzfile(file.path(out_dir,"tumor_only_balanced_cells.csv.gz")),row.names=FALSE)
data_use <- source_object@data[,selected,drop=FALSE]; meta_use <- selection[,c("ccc_group","sample","status"),drop=FALSE]; rm(source_object); invisible(gc())
cellchat_balanced <- createCellChat(object=data_use,meta=meta_use,group.by="ccc_group")
cellchat_balanced@DB <- CellChatDB.human; cellchat_balanced <- subsetData(cellchat_balanced)
cellchat_balanced <- identifyOverExpressedGenes(cellchat_balanced); cellchat_balanced <- identifyOverExpressedInteractions(cellchat_balanced)
lr_use <- cellchat_balanced@LR$LRsig[cellchat_balanced@LR$LRsig$ligand %in% candidate_ligands & cellchat_balanced@LR$LRsig$annotation=="Secreted Signaling",,drop=FALSE]
cellchat_balanced <- computeCommunProb(cellchat_balanced,type="triMean",LR.use=lr_use,population.size=FALSE,distance.use=FALSE,nboot=100,seed.use=seed)
cellchat_balanced <- filterCommunication(cellchat_balanced,min.cells=10)
communication <- subsetCommunication(cellchat_balanced); communication$source <- as.character(communication$source); communication$target <- as.character(communication$target)
focus <- communication[communication$source %in% source_states & communication$target %in% cm1_nodes & communication$annotation=="Secreted Signaling",,drop=FALSE]
write.csv(communication,file.path(out_dir,"tumor_only_balanced_cellchat_all_communication.csv"),row.names=FALSE)
write.csv(focus,file.path(out_dir,"tumor_only_epi_jun_vim_to_cm1_secreted.csv"),row.names=FALSE)
saveRDS(cellchat_balanced,file.path(out_dir,"tumor_only_balanced_cellchat_object.rds"),compress=FALSE)
future::plan("sequential")


In [ ]:
from matplotlib.patches import FancyArrowPatch

OUT_C = FIGURE_ROOT / "figure5c_cellchat_all_genes"
OUT_C.mkdir(parents=True, exist_ok=True)
CELLCHAT_CSV = OUT_C / "tumor_only_cellchat/tumor_only_epi_jun_vim_to_cm1_secreted.csv"
plt.rcdefaults()
plt.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none","font.family":"DejaVu Sans","font.size":8.5,"axes.titlesize":10.5,"axes.labelsize":8.5,"xtick.labelsize":7.2,"ytick.labelsize":7.2,"axes.linewidth":0.7})
CM1_ORDER = ['T_IL7R', 'T_GNLY', 'T_KLRB1', 'T_FOXP3', 'Mast_FYN_high', 'S_DCN', 'T_TOX', 'T_GZMK', 'NK_KLRF1', 'NK_XCL1']
PAIR_ORDER = ['SPP1|CD44', 'SPP1|ITGA4_ITGB1', 'SPP1|ITGAV_ITGB1', 'SPP1|ITGAV_ITGB5']
ROUTE_COLORS = {'vim': '#2A9D8F', 'jun': '#E67E22', 'cm1': '#3D6D9C', 'spp1': '#C94F49', 'dark': '#263238', 'mid': '#66717B', 'gray': '#9AA3AA'}

def save_route_pdf(fig: plt.Figure, stem: Path, dpi: int=400) -> None:
    fig.savefig(stem.with_suffix('.pdf'), bbox_inches='tight')
    plt.close(fig)

def route_header(fig: plt.Figure, panel: str, title: str, subtitle: str) -> None:
    fig.text(0.025, 0.955, panel, ha='left', va='top', fontsize=15, fontweight='bold')
    fig.text(0.085, 0.955, title, ha='left', va='top', fontsize=12, fontweight='bold', color=ROUTE_COLORS['dark'])
    fig.text(0.085, 0.9, subtitle, ha='left', va='top', fontsize=8.5, color=ROUTE_COLORS['mid'])

def prepare_route_source() -> tuple[pd.DataFrame, pd.DataFrame]:
    data = pd.read_csv(CELLCHAT_CSV)
    data['lr_pair'] = data['ligand'].astype(str) + '|' + data['receptor'].astype(str)
    data = data.loc[data['lr_pair'].isin(PAIR_ORDER)].copy()
    grouped = data.groupby(['source', 'target', 'lr_pair'], observed=True)['prob'].sum().rename('prob').reset_index()
    grid = pd.MultiIndex.from_product([['Epi_JUN', 'Epi_VIM'], CM1_ORDER, PAIR_ORDER], names=['source', 'target', 'lr_pair']).to_frame(index=False)
    grouped = grid.merge(grouped, how='left', on=['source', 'target', 'lr_pair'])
    grouped['prob'] = grouped['prob'].fillna(0.0)
    wide = grouped.pivot(index=['target', 'lr_pair'], columns='source', values='prob').reset_index()
    wide.columns.name = None
    wide['max_probability'] = wide[['Epi_JUN', 'Epi_VIM']].max(axis=1)
    epsilon = 0.0001
    wide['log2_jun_vim_probability_ratio'] = np.log2((wide['Epi_JUN'] + epsilon) / (wide['Epi_VIM'] + epsilon))
    wide['display_log2_ratio'] = wide['log2_jun_vim_probability_ratio'].clip(-1.0, 1.0)
    wide.to_csv(OUT_C / 'tumor_only_cellchat_differential_bubble_source_data.csv', index=False)
    aggregate = grouped.groupby(['source', 'lr_pair'], observed=True)['prob'].sum().unstack('source', fill_value=0).reindex(PAIR_ORDER).reset_index()
    aggregate['jun_vim_ratio'] = (aggregate['Epi_JUN'] + epsilon) / (aggregate['Epi_VIM'] + epsilon)
    aggregate['log2_jun_vim_ratio'] = np.log2(aggregate['jun_vim_ratio'])
    aggregate.to_csv(OUT_C / 'tumor_only_cellchat_pair_aggregate_source_data.csv', index=False)
    return (wide, aggregate)

def line_arrow(ax: plt.Axes, start: tuple[float, float], end: tuple[float, float], *, color: str, width: float, alpha: float=0.75, bend: float=0.0, square: bool=False, shrink_a: float=7.0, shrink_b: float=7.0, linestyle: str | tuple='-') -> None:
    ax.add_patch(FancyArrowPatch(start, end, connectionstyle='arc3,rad=0' if square else f'arc3,rad={bend}', arrowstyle='-|>', mutation_scale=9, lw=width, color=color, alpha=alpha, linestyle=linestyle, shrinkA=shrink_a, shrinkB=shrink_b, capstyle='butt' if square else 'round', joinstyle='miter' if square else 'round', zorder=1))

def edge_value_label(ax: plt.Axes, start: tuple[float, float], end: tuple[float, float], value: float, *, fraction: float=0.5, offset: tuple[float, float]=(0.0, 0.0), color: str) -> None:
    x = start[0] + fraction * (end[0] - start[0]) + offset[0]
    y = start[1] + fraction * (end[1] - start[1]) + offset[1]
    ax.text(x, y, f'{value:.3f}', ha='center', va='center', color=color, fontsize=7.4, fontweight='bold', bbox={'boxstyle': 'square,pad=0.12', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.88}, zorder=6)

def plot_route(wide: pd.DataFrame, *, straight_square: bool=False, annotate_scores: bool=False, vim_source_dashed: bool=False, show_route_legend: bool=False, output_stem: str | None=None) -> None:
    receptor_label = {'SPP1|CD44': 'CD44', 'SPP1|ITGA4_ITGB1': 'α4β1', 'SPP1|ITGAV_ITGB1': 'αVβ1', 'SPP1|ITGAV_ITGB5': 'αVβ5'}
    data = wide.copy()
    data['receiver_group'] = np.where(data['target'].str.startswith(('T_', 'NK_')), 'T/NK CM1 states', np.where(data['target'].eq('S_DCN'), 'Stromal DCN', 'Mast FYN-high'))
    route = data.groupby(['lr_pair', 'receiver_group'])[['Epi_JUN', 'Epi_VIM']].sum().reset_index()
    route.to_csv(OUT_C / 'Figure5C_all_genes_aggregated_CellChat_routes.csv', index=False)
    source_totals = data[['Epi_JUN', 'Epi_VIM']].sum()
    ratio = float(source_totals['Epi_JUN'] / source_totals['Epi_VIM'])
    (fig, ax) = plt.subplots(figsize=(7.09, 4.65))
    fig.subplots_adjust(left=0.025, right=0.98, bottom=0.08, top=0.8)
    route_header(fig, 'C', 'Epi_JUN supplies a stronger predicted SPP1 route to CM1 states', 'Tumour-only balanced CellChat; labels show aggregated probability and widths are scaled within stage' if annotate_scores else 'Tumour-only balanced CellChat from the all-gene source; line width encodes aggregated probability')
    if show_route_legend:
        legend_handles = [Line2D([0], [0], color=ROUTE_COLORS['jun'], lw=2.6, linestyle='-', label='Epi_JUN total'), Line2D([0], [0], color=ROUTE_COLORS['vim'], lw=2.6, linestyle='--', label='Epi_VIM total'), Line2D([0], [0], color=ROUTE_COLORS['spp1'], lw=2.2, linestyle='-', label='JUN: SPP1→receptor'), Line2D([0], [0], color=ROUTE_COLORS['cm1'], lw=2.2, linestyle='-', label='JUN: receptor→receiver')]
        fig.legend(handles=legend_handles, loc='upper left', bbox_to_anchor=(0.085, 0.845), frameon=False, ncol=4, handlelength=2.0, columnspacing=1.15, handletextpad=0.45, fontsize=7.1, borderaxespad=0.0)
    source_pos = {'Epi_JUN': (0.09, 0.67), 'Epi_VIM': (0.09, 0.33)}
    spp1_pos = (0.34, 0.5)
    receptor_pos = {'SPP1|CD44': (0.56, 0.79), 'SPP1|ITGA4_ITGB1': (0.56, 0.59), 'SPP1|ITGAV_ITGB1': (0.56, 0.39), 'SPP1|ITGAV_ITGB5': (0.56, 0.19)}
    receiver_pos = {'T/NK CM1 states': (0.82, 0.7), 'Stromal DCN': (0.82, 0.37), 'Mast FYN-high': (0.82, 0.16)}
    max_source = float(source_totals.max())
    for (source, position) in source_pos.items():
        color = ROUTE_COLORS['jun'] if source == 'Epi_JUN' else ROUTE_COLORS['vim']
        width = 2.2 + 6.0 * float(source_totals[source]) / max_source
        line_arrow(ax, position, spp1_pos, color=color, width=width, alpha=1.0, bend=0 if straight_square else -0.08 if source == 'Epi_JUN' else 0.08, square=straight_square, shrink_a=13, shrink_b=15, linestyle='--' if vim_source_dashed and source == 'Epi_VIM' else '-')
        ax.scatter(*position, s=620, color=color, edgecolor='none', linewidth=0, zorder=4)
        ax.text(*position, source.replace('Epi_', 'Epi\n'), ha='center', va='center', color='white', fontsize=10, fontweight='bold', zorder=5)
        if annotate_scores:
            source_offset = (0.0, 0.035 if source == 'Epi_JUN' else -0.035)
            edge_value_label(ax, position, spp1_pos, float(source_totals[source]), fraction=0.48, offset=source_offset, color=color)
    ax.scatter(*spp1_pos, s=760, color=ROUTE_COLORS['spp1'], edgecolor='none', linewidth=0, zorder=4)
    ax.text(*spp1_pos, 'SPP1', ha='center', va='center', color='white', fontsize=11, fontweight='bold', zorder=5)
    ax.text(0.205, 0.91, f'{ratio:.2f}× stronger total output', ha='center', color=ROUTE_COLORS['spp1'], fontweight='bold', fontsize=9)
    receptor_totals = data.groupby('lr_pair')['Epi_JUN'].sum()
    max_receptor = float(receptor_totals.max())
    for (pair, position) in receptor_pos.items():
        width = 1.1 + 5.5 * float(receptor_totals[pair]) / max_receptor
        line_arrow(ax, spp1_pos, position, color=ROUTE_COLORS['spp1'], width=width, alpha=1.0, square=straight_square, shrink_a=15, shrink_b=11)
        ax.scatter(*position, s=370, color='white', edgecolor=ROUTE_COLORS['spp1'], linewidth=1.7, zorder=4)
        ax.text(*position, receptor_label[pair], ha='center', va='center', color=ROUTE_COLORS['dark'], fontsize=9, fontweight='bold', zorder=5)
        if annotate_scores:
            receptor_offsets = {'SPP1|CD44': (-0.008, 0.022), 'SPP1|ITGA4_ITGB1': (-0.005, 0.024), 'SPP1|ITGAV_ITGB1': (-0.005, -0.024), 'SPP1|ITGAV_ITGB5': (-0.005, -0.024)}
            edge_value_label(ax, spp1_pos, position, float(receptor_totals[pair]), fraction=0.52, offset=receptor_offsets[pair], color=ROUTE_COLORS['spp1'])
    max_route = float(route['Epi_JUN'].max())
    for (_, row) in route.loc[route['Epi_JUN'] > 0].iterrows():
        route_start = receptor_pos[row['lr_pair']]
        route_end = receiver_pos[row['receiver_group']]
        line_arrow(ax, route_start, route_end, color=ROUTE_COLORS['cm1'], width=0.9 + 4.0 * float(row['Epi_JUN']) / max_route, alpha=1.0, square=straight_square, shrink_a=11, shrink_b=9)
        if annotate_scores:
            route_key = (row['lr_pair'], row['receiver_group'])
            route_label_specs = {('SPP1|CD44', 'T/NK CM1 states'): (0.48, (0.0, 0.025)), ('SPP1|CD44', 'Stromal DCN'): (0.56, (0.012, 0.016)), ('SPP1|CD44', 'Mast FYN-high'): (0.54, (0.012, -0.025)), ('SPP1|ITGA4_ITGB1', 'T/NK CM1 states'): (0.47, (0.0, -0.026)), ('SPP1|ITGAV_ITGB1', 'Stromal DCN'): (0.48, (0.0, 0.024)), ('SPP1|ITGAV_ITGB5', 'Stromal DCN'): (0.48, (0.0, -0.025))}
            (fraction, offset) = route_label_specs[route_key]
            edge_value_label(ax, route_start, route_end, float(row['Epi_JUN']), fraction=fraction, offset=offset, color=ROUTE_COLORS['cm1'])
    for (receiver, position) in receiver_pos.items():
        color = ROUTE_COLORS['cm1'] if receiver == 'T/NK CM1 states' else ROUTE_COLORS['gray']
        ax.scatter(*position, s=260, color=color, edgecolor='none', linewidth=0, zorder=4)
        ax.text(position[0] + 0.035, position[1], receiver, ha='left', va='center', color=ROUTE_COLORS['dark'], fontsize=8.7, fontweight='bold', zorder=5)
    ax.text(0.56, 0.94, 'SPP1 receptors', ha='center', color=ROUTE_COLORS['mid'], fontsize=8.5, fontweight='bold')
    ax.text(0.86, 0.94, 'Receiver groups', ha='center', color=ROUTE_COLORS['mid'], fontsize=8.5, fontweight='bold')
    if annotate_scores:
        ax.text(0.56, 0.035, 'Source labels show total output; downstream labels show Epi_JUN route probability', ha='center', va='center', color=ROUTE_COLORS['mid'], fontsize=7.4)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    stem = output_stem or ('Figure5C_all_genes_SPP1_route_schematic_straight_node_edge_aligned_no_white_outline_opaque' if straight_square else 'Figure5C_all_genes_SPP1_route_schematic')
    save_route_pdf(fig, OUT_C / stem)

wide, route_aggregate = prepare_route_source()
plot_route(wide, straight_square=True, annotate_scores=True, vim_source_dashed=True, show_route_legend=True,
           output_stem="Figure5C_all_genes_SPP1_route_schematic_straight_node_edge_aligned_no_white_outline_opaque_edge_scores_vim_dashed_legend")


## 7. Final Figure 5D: spatial SPP1-receptor hotspot source data and map

The fixed Tangram CM1/Epi_JUN spot scores are aligned to each spatial expression matrix. SPP1 and receptor gene-set scores are z-scored within section and converted to percentile ranks before defining top-quartile hotspots.


In [ ]:
import math
import scanpy as sc
from scipy import stats as scipy_stats

SPATIAL_ROOT = Path("/mnt/disk18t/lr_xcy/riku/codex_research/CM_analysis_2/spatial")
TANGRAM_TABLES = SPATIAL_ROOT / "cm_assignment_analysis_method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells/tables"
SPATIAL_H5AD = SPATIAL_ROOT / "cm_assignment_analysis/assign_nonepi_cm/mendeley_nonepi_cm_h5ad_9x9KL"
METHOD = "method16_tangram_pseudobulk_subtype_mapping_score_top100_all_cells_use_X"
SECTIONS = ["h46t","j38t","r114t","r29t","r51t","s15t","x49t","x98t","y12t","y27t","y7t","z43t"]
RECEPTORS = ["CD44","ITGAV","ITGB1","ITGA4","ITGA5","ITGB5"]
OUT_D = FIGURE_ROOT / "figure5d_spatial_hotspot_maps_background_legend"; OUT_D.mkdir(parents=True,exist_ok=True)

def percentile(values):
    series = pd.to_numeric(pd.Series(values), errors="coerce")
    result = pd.Series(np.nan, index=series.index, dtype=float)
    valid = series.dropna()
    if not valid.empty:
        result.loc[valid.index] = 0.0 if valid.nunique() <= 1 else valid.rank(method="average", pct=True)
    return result.to_numpy()
def gene_score(adata, genes):
    present=[gene for gene in genes if gene in adata.var_names]
    if not present: return np.full(adata.n_obs,np.nan)
    x=adata[:,present].X; x=x.toarray() if sparse.issparse(x) else np.asarray(x); x=np.asarray(x,dtype=float); sd=np.nanstd(x,axis=0); sd[~np.isfinite(sd) | (sd==0)]=np.nan
    return np.nanmean((x-np.nanmean(x,axis=0))/sd,axis=1)

SPATIAL_COLORS = {'ca9': '#6B5CA5', 'vim': '#2A9D8F', 'jun': '#E67E22', 'cm1': '#3D6D9C', 'spp1': '#C94F49', 'purple': '#7651A8', 'dark': '#263238', 'mid': '#66717B', 'light': '#E9EDF0', 'gray': '#9AA3AA'}

def spatial_configure() -> None:
    regular = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arial.TTF')
    bold = Path('/home/lr_xcy/.local/share/fonts/codex_trajectory_20260813/Arialbd.TTF')
    for path in (regular, bold):
        if path.exists():
            font_manager.fontManager.addfont(path)
    mpl.rcParams.update({'font.family': 'Arial' if regular.exists() else 'DejaVu Sans', 'font.size': 8.5, 'axes.titlesize': 9.5, 'axes.labelsize': 9.0, 'xtick.labelsize': 8.0, 'ytick.labelsize': 8.0, 'legend.fontsize': 8.0, 'axes.linewidth': 0.75, 'xtick.major.width': 0.75, 'ytick.major.width': 0.75, 'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none', 'figure.facecolor': 'white', 'savefig.facecolor': 'white'})

def spatial_header(fig: plt.Figure, letter: str, title: str, subtitle: str) -> None:
    fig.text(0.035, 0.965, letter, fontsize=16, fontweight='bold', color=SPATIAL_COLORS['dark'], va='top')
    fig.text(0.095, 0.965, title, fontsize=12.5, fontweight='bold', color=SPATIAL_COLORS['dark'], va='top')
    fig.text(0.095, 0.91, subtitle, fontsize=8.5, color=SPATIAL_COLORS['mid'], va='top')

plt.rcdefaults()
spatial_configure()

spot_tables=[]; section_rows=[]
for sample in SECTIONS:
    score_path=TANGRAM_TABLES / f"{sample}_{METHOD}_spot_scores.csv"; h5ad_path=SPATIAL_H5AD / f"adata_{sample}_CM.h5ad"
    score=pd.read_csv(score_path,index_col=0); adata_sp=sc.read_h5ad(h5ad_path); common=adata_sp.obs_names.intersection(score.index); adata_sp=adata_sp[common].copy(); score=score.reindex(common).copy()
    for column in ["array_row","array_col"]:
        if column not in score and column in adata_sp.obs: score[column]=adata_sp.obs[column].to_numpy()
    score["CM1_pct"]=percentile(score["CMact__joint_01_sharedCM"]); score["Epi_JUN_pct"]=percentile(score["EPIfrac__Epi_JUN"]); score["JUN_CM1_double_high"]=score[["CM1_pct","Epi_JUN_pct"]].min(axis=1)
    score["SPP1_ligand_score"]=gene_score(adata_sp,["SPP1"]); score["SPP1_receptor_score"]=gene_score(adata_sp,RECEPTORS)
    score["SPP1_ligand_pct"]=percentile(score["SPP1_ligand_score"]); score["SPP1_receptor_pct"]=percentile(score["SPP1_receptor_score"]); score["SPP1_LR_double_high"]=score[["SPP1_ligand_pct","SPP1_receptor_pct"]].min(axis=1)
    score["sample"]=sample; score["spot_id"]=score.index; rho,p=scipy_stats.spearmanr(score.JUN_CM1_double_high,score.SPP1_LR_double_high,nan_policy="omit"); section_rows.append({"sample":sample,"n_spots":len(score),"spearman_rho":rho,"p_value":p}); spot_tables.append(score.reset_index(drop=True))
all_spots=pd.concat(spot_tables,ignore_index=True); spatial_summary=pd.DataFrame(section_rows)
all_spots.to_csv(FINAL_SOURCE / "Figure5D_spatial_hotspot_spot_source_data.csv.gz",index=False,compression="gzip"); spatial_summary.to_csv(FINAL_SOURCE / "Figure5D_spatial_hotspot_section_summary.csv",index=False)

data=all_spots[all_spots["sample"].eq("x98t")].copy(); niche=data.JUN_CM1_double_high.to_numpy()>=.75; route=data.SPP1_LR_double_high.to_numpy()>=.75; overlap=niche & route
fig,axes=plt.subplots(1,3,figsize=(7.09,3.60)); fig.subplots_adjust(left=.025,right=.98,bottom=.08,top=.70,wspace=.04); n_positive=int((spatial_summary.spearman_rho>0).sum())
spatial_header(fig,"D","SPP1-receptor hotspots fall within JUN–CM1-rich tissue regions",f"Representative x98t section; top-quartile hotspots; section-level association is positive in {n_positive}/{len(spatial_summary)} sections")
hotspot_colors={"background":"#E8ECEF","niche":"#7651A8","route":"#C94F49","overlap":"#5B2E73"}
fig.legend(handles=[Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["background"],markeredgecolor="none",markersize=5.4,label="Background spots"),Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["niche"],markeredgecolor="none",markersize=5.4,label="JUN–CM1 hotspot"),Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["route"],markeredgecolor="none",markersize=5.4,label="SPP1–receptor hotspot"),Line2D([0],[0],marker="o",linestyle="none",markerfacecolor=hotspot_colors["overlap"],markeredgecolor="none",markersize=5.4,label="Overlap hotspot")],loc="upper center",bbox_to_anchor=(.52,.825),frameon=False,ncol=4,columnspacing=1.35,handletextpad=.35,borderaxespad=0.,fontsize=7.5)
x=data.array_col.to_numpy(float); y=-data.array_row.to_numpy(float)*math.sqrt(3)
for ax,mask,color,title in zip(axes,[niche,route,overlap],[hotspot_colors["niche"],hotspot_colors["route"],hotspot_colors["overlap"]],["JUN–CM1 hotspot","SPP1–receptor route",f"Spatial overlap\n{int(overlap.sum())} spots"]):
    ax.scatter(x,y,s=4,color=hotspot_colors["background"],linewidth=0,alpha=.85,rasterized=True); ax.scatter(x[mask],y[mask],s=9,color=color,linewidth=0,alpha=.98,rasterized=True); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title,color=color,fontsize=9.5,fontweight="bold",pad=5)
fig.savefig(OUT_D / "Figure5D_spatial_hotspot_maps_with_background_legend.pdf",dpi=450,bbox_inches="tight"); plt.close(fig)
